In [ ]:
"""
Machine Learning Milestone: Analyzing what drives developer salaries.

In this stage, we're comparing a standard baseline model against an extended version that accounts for TIOBE language popularity. The goal is to determine if 'trending' languages have a measurable impact on pay. All resulting comparison charts and feature importance tables are automatically exported to the outputs folder for our GitHub repo.
"""



#First,let's make sure we have a home for our data and results.
#These lines ensure the folders exist so the script doesn't crash when saving later.
import os

os.makedirs("data/processed", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)


from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#ML Toolkit
#Bringing in the heavy lifters for our analysis:
#We'll use Ridge for a solid baseline and Random Forest to catch more complex patterns.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer


warnings.filterwarnings("ignore")


RANDOM_STATE = 42

DATA_PATH = Path("data/processed/language_salary_with_tiobe_sample.csv")
OUTPUT_TABLES = Path("outputs/tables")
OUTPUT_FIGURES = Path("outputs/figures")




In [ ]:
def clean_numeric_experience(value):
    """Convert Stack Overflow experience strings into numeric values."""
    #If the data is missing, we'll keep it as NaN to avoid skewing results
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    #Handling the 'edge cases' defined by the survey:
    #'Less than 1 year' is treated as a 0.5 year average for modeling

    if value.lower() in ["less than 1 year", "less than 1 years"]:
        return 0.5
    #For the veterans with 50+ years, we'll cap it at 51 to keep it numeric
    if value.lower() in ["more than 50 years", "more than 50"]:
        return 51

    #If it's a standard number, convert to float; if it's weird text, mark as NaN
    try:
        return float(value)
    except ValueError:
        return np.nan


def load_and_prepare_data(path):
    """Load processed dataset and keep variables needed for ML modeling."""
    df = pd.read_csv(path)

    #Convert experience variables safely.
    df["YearsCodePro_num"] = df["YearsCodePro"].apply(clean_numeric_experience)
    df["YearsCode_num"] = df["YearsCode"].apply(clean_numeric_experience)

    #Keep rows with valid target and core predictors.
    df = df.dropna(subset=["ConvertedCompYearly", "LogSalary", "YearsCodePro_num"])

    #Salary data is usually skewed. The target is already log-transformed in the EDA pipeline.
    #We still keep the original salary for reporting MAE/RMSE in USD after inverse transformation.
    df = df[df["ConvertedCompYearly"] > 0].copy()

    #Fill missing popularity with 0 for languages not covered by the TIOBE table.
    df["Avg_TIOBE_Rating"] = df["Avg_TIOBE_Rating"].fillna(0)
    df["Months_Observed"] = df["Months_Observed"].fillna(0)

    #Missing categorical values become explicit categories.
    categorical_candidates = [
        "Age",
        "RemoteWork",
        "EdLevel",
        "DevType",
        "OrgSize",
        "Country",
        "Language",
        "Industry",
        "ProfessionalCloud",
        "AISelect",
    ]

    for col in categorical_candidates:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown").astype(str)

    return df


def build_preprocessor(numeric_features, categorical_features):
    """Create preprocessing pipeline for numeric and categorical variables."""
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]
    )

    categorical_transformer = OneHotEncoder(
        handle_unknown="ignore",
        max_categories=30
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )


def evaluate_model(name, pipeline, X_train, X_test, y_train, y_test):
    """Train a model and return log-scale and dollar-scale performance metrics."""
    pipeline.fit(X_train, y_train)
    pred_log = pipeline.predict(X_test)

    #The EDA pipeline used natural log salary, so exp() maps back to yearly compensation.
    y_test_usd = np.exp(y_test)
    pred_usd = np.exp(pred_log)

    return {
        "Model": name,
        "MAE_log": mean_absolute_error(y_test, pred_log),
        "RMSE_log": np.sqrt(mean_squared_error(y_test, pred_log)),
        "R2_log": r2_score(y_test, pred_log),
        "MAE_USD": mean_absolute_error(y_test_usd, pred_usd),
        "RMSE_USD": np.sqrt(mean_squared_error(y_test_usd, pred_usd)),
    }, pipeline


def get_feature_importance(fitted_pipeline, top_n=25):
    """Extract feature importance from a fitted tree-based pipeline."""
    preprocessor = fitted_pipeline.named_steps["preprocess"]
    model = fitted_pipeline.named_steps["model"]

    feature_names = preprocessor.get_feature_names_out()
    importances = model.feature_importances_

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values("Importance", ascending=False)

    return importance_df.head(top_n)




In [ ]:
def main():
    OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
    OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)

    df = load_and_prepare_data(DATA_PATH)

    #Baseline: structural variables only.
    baseline_numeric = ["YearsCodePro_num", "YearsCode_num"]
    baseline_categorical = ["Age", "RemoteWork", "EdLevel", "DevType", "OrgSize", "Country"]

    #Extended: structural variables + language identity + language popularity information.
    extended_numeric = baseline_numeric + ["Avg_TIOBE_Rating", "Months_Observed"]
    extended_categorical = baseline_categorical + ["Language"]

    y = df["LogSalary"]

    #Group split prevents the same developer appearing in both train and test
    #after the dataset was expanded by programming language.
    groups = df["ResponseId"] if "ResponseId" in df.columns else np.arange(len(df))
    splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(df, y, groups=groups))

    results = []
    fitted_models = {}

    model_specs = [
        (
            "Baseline Ridge Regression",
            baseline_numeric,
            baseline_categorical,
            Ridge(alpha=1.0)
        ),
        (
            "Baseline Random Forest",
            baseline_numeric,
            baseline_categorical,
            RandomForestRegressor(
                n_estimators=60,
                max_depth=10,
                min_samples_leaf=5,
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        ),
        (
            "Extended Ridge Regression + TIOBE",
            extended_numeric,
            extended_categorical,
            Ridge(alpha=1.0)
        ),
        (
            "Extended Random Forest + TIOBE",
            extended_numeric,
            extended_categorical,
            RandomForestRegressor(
                n_estimators=60,
                max_depth=10,
                min_samples_leaf=5,
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        ),

    ]

    for name, numeric_features, categorical_features, model in model_specs:
        X = df[numeric_features + categorical_features]
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        pipe = Pipeline(
            steps=[
                ("preprocess", build_preprocessor(numeric_features, categorical_features)),
                ("model", model)
            ]
        )

        metrics, fitted_pipe = evaluate_model(name, pipe, X_train, X_test, y_train, y_test)
        results.append(metrics)
        fitted_models[name] = fitted_pipe

    results_df = pd.DataFrame(results).sort_values("R2_log", ascending=False)
    results_df.to_csv(OUTPUT_TABLES / "ml_model_comparison.csv", index=False)

    #Feature importance from the main proposed model.
    rf_model_name = "Extended Random Forest + TIOBE"
    importance_df = get_feature_importance(fitted_models[rf_model_name])
    importance_df.to_csv(OUTPUT_TABLES / "ml_feature_importance_random_forest.csv", index=False)

    #Plot 1: model comparison by R2.
    plt.figure(figsize=(10, 5))
    plt.bar(results_df["Model"], results_df["R2_log"])
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("R² on log salary")
    plt.title("ML Model Comparison")
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURES / "ml_model_r2_comparison.png", dpi=200)
    plt.close()

    #Plot 2: top feature importance.
    plot_importance = importance_df.sort_values("Importance", ascending=True)
    plt.figure(figsize=(10, 7))
    plt.barh(plot_importance["Feature"], plot_importance["Importance"])
    plt.xlabel("Importance")
    plt.title("Top Feature Importances - Extended Random Forest")
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURES / "ml_top_feature_importance.png", dpi=200)
    plt.close()

    print("\nMachine Learning results saved successfully.")
    print("\nModel comparison:")
    print(results_df.to_string(index=False))
    print("\nTop Random Forest features:")
    print(importance_df.to_string(index=False))


if __name__ == "__main__":
    main()



Machine Learning results saved successfully.

Model comparison:
                            Model  MAE_log  RMSE_log   R2_log      MAE_USD     RMSE_USD
Extended Ridge Regression + TIOBE 0.553910  0.874899 0.481235 29946.408451 46476.547014
        Baseline Ridge Regression 0.555473  0.878985 0.476379 29988.649379 46419.792179
           Baseline Random Forest 0.565263  0.882891 0.471714 30234.593912 45898.999774
   Extended Random Forest + TIOBE 0.565685  0.884289 0.470041 30206.656627 46079.954408

Top Random Forest features:
                                                            Feature  Importance
                              cat__Country_United States of America    0.268313
                                                 num__YearsCode_num    0.170530
                                              num__YearsCodePro_num    0.165962
                                    cat__Country_infrequent_sklearn    0.061838
                                               cat__Country_Ukrain